In [ ]:
import os, zipfile, urllib.request
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 60)

In [ ]:
#Load the dataset
URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

if not os.path.exists("ml-latest-small"):
    urllib.request.urlretrieve(URL, "ml-latest-small.zip")
    with zipfile.ZipFile("ml-latest-small.zip") as z:
        z.extractall(".")

movies = pd.read_csv("ml-latest-small/movies.csv")
ratings = pd.read_csv("ml-latest-small/ratings.csv")

print(f"{len(movies)} movies, {len(ratings)} ratings, {ratings.userId.nunique()} users")
display(movies.head())
display(ratings.head())

9742 movies, 100836 ratings, 610 users


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [ ]:
#Popularity stats and helper
stats = ratings.groupby("movieId").agg(
    n_ratings=("rating", "count"),
    avg_rating=("rating", "mean"),
).reset_index()

movies = movies.merge(stats, on="movieId", how="left")
movies["n_ratings"] = movies["n_ratings"].fillna(0)
movies["avg_rating"] = movies["avg_rating"].fillna(0)

C = ratings["rating"].mean()                 # global mean rating
m = movies["n_ratings"].quantile(0.80)       # vote threshold

movies["weighted_rating"] = (
    movies["n_ratings"] / (movies["n_ratings"] + m) * movies["avg_rating"]
    + m / (movies["n_ratings"] + m) * C
)

title_by_id = movies.set_index("movieId")["title"]


def find_movie(query):
    # Returns the most-rated movie whose title contains the query
    hits = movies[movies["title"].str.contains(query, case=False, regex=False)]
    if hits.empty:
        raise ValueError(f"No movie matching '{query}'")
    return hits.sort_values("n_ratings", ascending=False).iloc[0]


print(f"Global mean rating: {C:.2f}, vote threshold m: {m:.0f}")

Global mean rating: 3.50, vote threshold m: 12


In [ ]:
#Content-based filtering
movies["genre_text"] = (
    movies["genres"]
    .replace("(no genres listed)", "")
    .str.replace("|", " ", regex=False)
)

tfidf = TfidfVectorizer(token_pattern=r"[^ ]+", lowercase=True)   # keeps "sci-fi", "film-noir" intact
genre_matrix = tfidf.fit_transform(movies["genre_text"])
print("TF-IDF matrix:", genre_matrix.shape)


def content_based(title, n=10, min_votes=10):
    movie = find_movie(title)
    print(f"Because you liked: {movie['title']}  [{movie['genres']}]\n")
    sims = cosine_similarity(genre_matrix[movie.name], genre_matrix).ravel()

    out = movies.assign(similarity=sims).copy()
    out = out[(out.index != movie.name) & (out["n_ratings"] >= min_votes)].copy()
    out["score"] = 0.7 * out["similarity"] + 0.3 * (out["weighted_rating"] / 5)

    cols = ["title", "genres", "similarity", "weighted_rating"]
    return out.sort_values("score", ascending=False)[cols].head(n).round(2)


display(content_based("Toy Story"))

TF-IDF matrix: (9742, 19)
Because you liked: Toy Story (1995)  [Adventure|Animation|Children|Comedy|Fantasy]



,title,genres,similarity,weighted_rating
3568,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,1.00,3.84
2355,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.00,3.82
3000,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.00,3.66
9430,Moana (2016),Adventure|Animation|Children|Comedy|Fantasy,1.00,3.48
8900,Inside Out (2015),Adventure|Animation|Children|Comedy|Drama|Fantasy,0.97,3.75
6944,Ponyo (Gake no ue no Ponyo) (2008),Adventure|Animation|Children|Fantasy,0.96,3.74
1706,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.00,3.30
6486,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,1.00,3.20
3194,Shrek (2001),Adventure|Animation|Children|Comedy|Fantasy|Romance,0.93,3.84
8357,The Lego Movie (2014),Action|Adventure|Animation|Children|Comedy|Fantasy,0.94,3.77


In [ ]:
#Collabrorative filtering model
def fit_item_cf(train, min_ratings=10):
    ui = train.pivot_table(index="userId", columns="movieId", values="rating")
    ui = ui.loc[:, ui.notna().sum() >= min_ratings]

    global_mean = train["rating"].mean()
    user_means = ui.mean(axis=1).fillna(global_mean)
    centered = ui.sub(user_means, axis=0).fillna(0.0)

    sim = cosine_similarity(centered.values.T)
    sim = np.clip(sim, 0, None)
    np.fill_diagonal(sim, 0)
    sim_df = pd.DataFrame(sim, index=ui.columns, columns=ui.columns)

    return {"ui": ui, "user_means": user_means, "centered": centered, "sim_df": sim_df}


def predict_all(model):
    # Predicted rating for every (user, movie) pair at once
    ui = model["ui"]
    sim = model["sim_df"].values
    rated = ui.notna().values.astype(float)

    num = model["centered"].values @ sim
    den = rated @ sim
    adj = np.divide(num, den, out=np.zeros_like(num), where=den > 0)

    pred = model["user_means"].values[:, None] + adj
    return pd.DataFrame(np.clip(pred, 0.5, 5.0), index=ui.index, columns=ui.columns)


cf = fit_item_cf(ratings)
pred_all = predict_all(cf)
print("Rating matrix:", cf["ui"].shape, "| predictions:", pred_all.shape)

Rating matrix: (610, 2269) | predictions: (610, 2269)


In [ ]:
#Similar movies and per-user recommendations
def similar_movies(title, n=10):
    # "Users who liked X also liked ..."
    movie = find_movie(title)
    mid = movie["movieId"]
    if mid not in cf["sim_df"].columns:
        print("Not enough ratings for this movie.")
        return None
    print(f"Movies rated similarly to: {movie['title']}\n")
    s = cf["sim_df"][mid].sort_values(ascending=False).head(n)
    return pd.DataFrame({
        "title": title_by_id.loc[s.index].values,
        "similarity": s.round(3).values,
    })


def recommend_for_user(user_id, n=10):
    if user_id not in pred_all.index:
        raise ValueError(f"Unknown user {user_id}")
    seen = cf["ui"].loc[user_id].notna()
    scores = pred_all.loc[user_id][~seen].sort_values(ascending=False).head(n)
    return pd.DataFrame({
        "title": title_by_id.loc[scores.index].values,
        "predicted_rating": scores.round(2).values,
    })


def user_favourites(user_id, n=5):
    r = ratings[ratings["userId"] == user_id].sort_values("rating", ascending=False).head(n)
    return pd.DataFrame({"title": title_by_id.loc[r["movieId"]].values, "rating": r["rating"].values})


display(similar_movies("Matrix, The"))

Movies rated similarly to: Matrix, The (1999)



,title,similarity
0,Fight Club (1999),0.367
1,Star Wars: Episode V - The Empire Strikes Back (1980),0.356
2,Saving Private Ryan (1998),0.340
3,Die Hard (1988),0.312
4,Reservoir Dogs (1992),0.312
5,Gattaca (1997),0.311
6,Star Wars: Episode IV - A New Hope (1977),0.301
7,Star Wars: Episode VI - Return of the Jedi (1983),0.297
8,Pulp Fiction (1994),0.289
9,Memento (2000),0.287


In [ ]:
#Recommend for a user
USER_ID = 1   # change this to any user id from 1 to 610

print(f"Highest-rated movies of user {USER_ID}:")
display(user_favourites(USER_ID))

print(f"Recommended for user {USER_ID}:")
display(recommend_for_user(USER_ID))

Highest-rated movies of user 1:


,title,rating
0,Seven (a.k.a. Se7en) (1995),5.0
1,"Usual Suspects, The (1995)",5.0
2,Bottle Rocket (1996),5.0
3,Dumb & Dumber (Dumb and Dumber) (1994),5.0
4,Billy Madison (1995),5.0


Recommended for user 1:


,title,predicted_rating
0,Six Degrees of Separation (1993),4.72
1,Superbad (2007),4.72
2,Blow (2001),4.72
3,3:10 to Yuma (2007),4.69
4,Family Guy Presents Stewie Griffin: The Untold Story (2005),4.69
5,Dirty Rotten Scoundrels (1988),4.69
6,Monty Python's And Now for Something Completely Differen...,4.68
7,North by Northwest (1959),4.68
8,Top Secret! (1984),4.68
9,"Producers, The (1968)",4.68


In [ ]:
#Evaluation (RMSE vs baseline)
train, test = train_test_split(ratings, test_size=0.2, random_state=42)

cf_train = fit_item_cf(train)
pred_train = predict_all(cf_train)

rows = pred_train.index.get_indexer(test["userId"])
cols = pred_train.columns.get_indexer(test["movieId"])
ok = (rows >= 0) & (cols >= 0)          # only movies/users the model knows about

y_true = test["rating"].values[ok]
y_cf = pred_train.values[rows[ok], cols[ok]]
y_base = cf_train["user_means"].reindex(test["userId"]).values[ok]


def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


print(f"Evaluated on {ok.sum()} of {len(test)} held-out ratings")
print(f"Baseline (user mean) RMSE : {rmse(y_true, y_base):.4f}")
print(f"Item-based CF RMSE        : {rmse(y_true, y_cf):.4f}")

Evaluated on 15050 of 20168 held-out ratings
Baseline (user mean) RMSE : 0.9245
Item-based CF RMSE        : 0.8349


In [ ]:
!pip install -q "gradio>=4,<6"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.6/325.6 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 462.4/462.4 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 2.12.1 requires pydantic<3.0.0,>=2.12.5, but you have pydantic 2.12.3 which is incompatible.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.14.0 which is incompatible.
google-adk 2.7.1 requires starlette<2,>=1.3.1, but you ha

In [ ]:
CSS = """
.gradio-container {
    background: linear-gradient(120deg,#fdf2f8,#ede9fe,#e0f2fe,#fef9c3,#fdf2f8) !important;
    background-size: 400% 400% !important;
    animation: bgmove 20s ease infinite;
}
@keyframes bgmove { 0%{background-position:0% 50%} 50%{background-position:100% 50%} 100%{background-position:0% 50%} }

.hero { text-align:center; padding:28px 10px 8px; }
.hero h1 { font-size:3rem; margin:0; color:#5b21b6 !important; text-shadow:none; }
.hero p  { font-size:1.15rem; margin-top:6px; color:#9d174d !important; opacity:.9; }

button[role="tab"] { color:#4c1d95 !important; font-weight:600; }
button[role="tab"].selected { color:#db2777 !important; }
footer, footer * { color:#6b21a8 !important; }

.banner { color:#4c1d95; font-size:1.1rem; margin:6px 2px 14px; }
.grid { display:grid; grid-template-columns:repeat(auto-fill,minmax(230px,1fr)); gap:16px; }
.card { position:relative; border-radius:18px; padding:18px 16px 14px; color:white;
        box-shadow:0 8px 20px rgba(91,33,182,.18); transition:transform .2s, box-shadow .2s; min-height:150px; }
.card:hover { transform:translateY(-6px) scale(1.02); box-shadow:0 14px 28px rgba(91,33,182,.30); }
.rank { position:absolute; top:10px; right:14px; font-weight:800; font-size:1.3rem; opacity:.55; }
.title { font-weight:700; font-size:1.05rem; margin:0 30px 10px 0; line-height:1.3; }
.chip { display:inline-block; padding:3px 10px; margin:0 5px 5px 0; border-radius:999px;
        font-size:.75rem; font-weight:600; color:white; border:1px solid rgba(255,255,255,.55); }
.score { margin-top:8px; font-size:.95rem; font-weight:700; background:rgba(0,0,0,.22);
         display:inline-block; padding:4px 12px; border-radius:12px; }
.msg { color:#4c1d95; background:white; padding:14px 18px; border-radius:14px;
       box-shadow:0 4px 12px rgba(91,33,182,.12); }
"""

In [ ]:
try:
    demo.close()          # stop the previous website if it is still running
except NameError:
    pass


def ui_similar(title, method):
    try:
        movie = find_movie(title)
    except ValueError:
        return message("Movie not found. Try another title, e.g. Toy Story or Matrix.")

    banner = f"<div class='banner'>Because you liked <b>{html.escape(movie['title'])}</b></div>"
    if method.startswith("Content"):
        df = content_based(title, 12)
        return banner + cards_html(df, "similarity", "🎯 match")
    df = similar_movies(title, 12)
    if df is None:
        return message("Not enough ratings for this movie. Try the content-based method.")
    return banner + cards_html(df, "similarity", "🎯 match")


def ui_genre(genre):
    df = top_by_genre(genre, 12)
    banner = f"<div class='banner'>Top rated <b>{html.escape(genre)}</b> movies</div>"
    return banner + cards_html(df, "weighted_rating", "⭐")


def ui_user(user_id):
    try:
        uid = int(user_id)
        favs = ", ".join(user_favourites(uid, 3)["title"])
        df = recommend_for_user(uid, 12)
    except (ValueError, TypeError):
        return message("Please enter a valid user id between 1 and 610.")
    banner = f"<div class='banner'>User {uid} loved: <b>{html.escape(favs)}</b></div>"
    return banner + cards_html(df, "predicted_rating", "⭐ predicted")


genre_choices = [g for g in all_genres if g != "(no genres listed)"]
CONTENT = "Content-based (genres)"
COLLAB = "Collaborative (user ratings)"

with gr.Blocks(css=CSS, theme=gr.themes.Soft(primary_hue="pink", secondary_hue="purple")) as demo:
    gr.HTML("<div class='hero'><h1>🎬 CineMatch</h1><p>Find your next favourite movie</p></div>")

    with gr.Tabs():
        with gr.Tab("🎞️ Similar movies"):
            with gr.Row():
                t_in = gr.Textbox(label="A movie you liked", value="Toy Story", scale=3)
                m_in = gr.Radio([CONTENT, COLLAB], value=CONTENT, label="Method", scale=2)
            b1 = gr.Button("✨ Recommend", variant="primary")
            o1 = gr.HTML()
            b1.click(ui_similar, [t_in, m_in], o1)
            t_in.submit(ui_similar, [t_in, m_in], o1)
            gr.Examples(
                examples=[
                    ["Toy Story", CONTENT], ["Toy Story", COLLAB],
                    ["Matrix, The", CONTENT], ["Matrix, The", COLLAB],
                    ["Dark Knight", COLLAB], ["Titanic", CONTENT],
                    ["Lion King", COLLAB], ["Pulp Fiction", COLLAB],
                    ["Forrest Gump", COLLAB], ["Finding Nemo", CONTENT],
                ],
                inputs=[t_in, m_in], outputs=o1, fn=ui_similar,
                run_on_click=True, cache_examples=False, label="👆 Click a sample to try it",
            )

        with gr.Tab("🎭 Browse by genre"):
            g_in = gr.Dropdown(genre_choices, value="Sci-Fi", label="Genre")
            b2 = gr.Button("🎨 Show top movies", variant="primary")
            o2 = gr.HTML()
            b2.click(ui_genre, g_in, o2)
            g_in.change(ui_genre, g_in, o2)
            gr.Examples(
                examples=[["Sci-Fi"], ["Comedy"], ["Horror"], ["Animation"],
                          ["Romance"], ["Thriller"], ["Documentary"]],
                inputs=g_in, outputs=o2, fn=ui_genre,
                run_on_click=True, cache_examples=False, label="👆 Click a sample genre",
            )

        with gr.Tab("👤 For a user"):
            u_in = gr.Number(label="User ID (1 to 610)", value=1, precision=0)
            b3 = gr.Button("🍿 Recommend for this user", variant="primary")
            o3 = gr.HTML()
            b3.click(ui_user, u_in, o3)
            gr.Examples(
                examples=[[1], [5], [414], [599]],
                inputs=u_in, outputs=o3, fn=ui_user,
                run_on_click=True, cache_examples=False, label="👆 Click a sample user",
            )

    # show sample results as soon as the page opens
    demo.load(ui_similar, [t_in, m_in], o1)
    demo.load(ui_genre, g_in, o2)
    demo.load(ui_user, u_in, o3)

demo.launch(share=True)

Closing server running on port: 7860


/tmp/ipykernel_1743/3297379073.py:44: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Soft(primary_hue="pink", secondary_hue="purple")) as demo:
/tmp/ipykernel_1743/3297379073.py:44: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Soft(primary_hue="pink", secondary_hue="purple")) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5d1ff5a159317ce4d7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
